# Lung Cancer Prediction with Logistic Regression

**Goal:** answer the questions a patient can actually answer (age, gender, smoking,
symptoms) and predict a **yes / no** diagnosis.

**Data:** `data/survey_lung_cancer.csv` — the UCI *Lung Cancer Survey* (309 respondents),
the standard dataset for this task. Every input column is a survey question and
`LUNG_CANCER` is a genuine YES/NO label.

> **Why not the first dataset?**
> `data/lung_cancer_dataset.csv` (2000 synthetic patients) has no diagnosis label —
> every row is already a diagnosed patient, and its only yes/no field, `Survived`,
> turned out to depend on post-diagnosis information (`Survival_Months` AUC 0.778,
> `Cancer_Stage` 0.697, `Treatment` 0.618). Restricting it to questionnaire inputs
> gave a model with ROC AUC 0.53 — pure noise. It is kept only as a reference.

**Pipeline:** clean -> explore -> train logistic regression -> evaluate -> save.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import lung_cancer_model as m

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

## 1. Load the raw survey

`m.load_raw()` reads the CSV and strips the stray whitespace the file ships in its
header (`"FATIGUE "`, `"ALLERGY "`).

In [ ]:
raw = m.load_raw()
print("raw shape:", raw.shape)
raw.head()

### What actually needs cleaning

The file is small but has four real problems:

1. **Header whitespace** — `CHRONIC DISEASE`, `FATIGUE `, `ALLERGY ` would otherwise
   become three differently-named columns.
2. **Encoded answers** — every question is `1` / `2`, where **2 means yes**. Reading
   this backwards silently inverts every conclusion, so it has to be explicit.
3. **33 duplicated rows** — 10% of the file is a verbatim repeat. Left in, the same
   respondent lands in both train and test and the scores are inflated.
4. **Severe class imbalance** — 86% of respondents are positive, so accuracy alone
   is meaningless: always answering "yes" scores 86%.

There is **no ID or serial column** here, so unlike a raw export there is nothing to
drop for being an identifier — all 15 columns are genuine questions.

In [ ]:
dirty = pd.read_csv(m.RAW_DATA)

print("headers with stray whitespace:", [c for c in dirty.columns if c != c.strip()])
print("SMOKING values          :", sorted(dirty["SMOKING"].unique()))
print("LUNG_CANCER values      :", sorted(dirty["LUNG_CANCER"].unique()))
print("GENDER values           :", sorted(dirty["GENDER"].unique()))
print("exact duplicated rows   :", int(dirty.duplicated().sum()))
print("share positive (YES)    :", round(float((dirty.LUNG_CANCER == "YES").mean()), 3))

## 2. Clean it

`m.clean_data()` does all four fixes and then **validates** the result: any unexpected
category (say a typo, or a `3`) raises instead of quietly becoming a constant column.

In [ ]:
clean = m.clean_data(raw, verbose=True)
print("cleaned shape:", clean.shape)
print("target balance:", clean[m.TARGET].value_counts().to_dict())
clean.head()

In [ ]:
print("missing values :", int(clean.isna().sum().sum()))
print()
print(clean.dtypes.to_string())

## 3. A quick look at the data

The imbalance is the single most important thing to see before modelling.

In [ ]:
counts = clean[m.TARGET].value_counts().sort_index()
ax = counts.rename({0: "No lung cancer", 1: "Lung cancer"}).plot(
    kind="bar", color=["#64748B", "#2563EB"], rot=0, figsize=(5, 3.5)
)
ax.set_title("Target balance: 86% of respondents are positive")
ax.set_ylabel("respondents")
ax.set_xlabel("")
plt.show()

### Which answers actually separate the two groups?

For each question, the share of respondents who were diagnosed. The dashed red line is
the overall positive rate (86%), so a bar *below* the line means that answering "yes"
was associated with **not** having cancer.

In [ ]:
rates = {
    col: clean.groupby(col)[m.TARGET].mean().get("Yes", np.nan)
    for col in m.BINARY_FEATURES
}
rate_series = pd.Series(rates).sort_values(ascending=False)

ax = rate_series.plot(kind="barh", figsize=(8, 6), color="#2563EB")
ax.invert_yaxis()
ax.axvline(clean[m.TARGET].mean(), color="#DC2626", linestyle="--",
           label="overall share positive")
ax.set_title("Share diagnosed, by answer")
ax.set_xlabel("share with lung cancer")
ax.set_ylabel("")
ax.legend()
plt.show()

rate_series.round(3).to_frame("share diagnosed")

In [ ]:
ax = sns.boxplot(data=clean, x=m.TARGET, y="Age", hue=m.TARGET,
                 palette="Blues", legend=False)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No", "Yes"])
ax.set_xlabel("lung cancer")
ax.set_title("Age barely separates the groups")
plt.show()

clean.groupby(m.TARGET)["Age"].describe().round(1)

Age looks like a weak signal — worth knowing before we read its coefficient.

## 4. Train the model

The pipeline (`m.build_pipeline()`) does three things before logistic regression:

* **`StandardScaler` on Age** — so its coefficient is *per standard deviation*, comparable
  with the yes/no answers.
* **`OrdinalEncoder(categories=[["No", "Yes"]])`** on the 13 questions — explicit order,
  No -> 0 and Yes -> 1.
* **`OneHotEncoder` on Gender** — no false ordering between the two values.

It also sets **`class_weight="balanced"`**. With only 14% negatives, an unweighted model
scores 86% by always answering "yes" while missing half the negative cases. Weighting
costs a little raw accuracy and buys a model that actually responds to its inputs.

`m.split_data()` is a *stratified* split, shared with `m.train()` so this notebook and
the training script can never drift apart.

In [ ]:
X_train, X_test, y_train, y_test = m.split_data(clean)
print("train:", X_train.shape, " test:", X_test.shape)
print("positive share  train:", round(float(y_train.mean()), 3),
      " test:", round(float(y_test.mean()), 3))

# m.train() re-creates exactly this split internally.
pipeline, metrics = m.train(clean)

## 5. Evaluate

**Accuracy is the wrong headline here.** The majority baseline is 0.855, so we judge the
model on ROC AUC (ranking quality) and balanced accuracy (treats both classes equally).

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "majority-class baseline accuracy",
            "test accuracy",
            "test balanced accuracy",
            "test ROC AUC",
            "5-fold CV ROC AUC",
            "5-fold CV balanced accuracy",
            "recall on 'No' (minority)",
            "recall on 'Yes'",
        ],
        "value": [
            metrics["majority_baseline"],
            metrics["test_accuracy"],
            metrics["test_balanced_accuracy"],
            metrics["test_roc_auc"],
            metrics["cv_roc_auc"],
            metrics["cv_balanced_accuracy"],
            metrics["recall_no"],
            metrics["recall_yes"],
        ],
    }
)
summary["value"] = summary["value"].round(3)
summary

In [ ]:
cm = np.array(metrics["confusion_matrix"])
ax = sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                 xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title(f"Held-out confusion matrix (accuracy {metrics['test_accuracy']:.3f})")
plt.show()

print(metrics["classification_report"])

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

y_prob = pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color="#2563EB", lw=2,
         label=f"logistic regression (AUC = {roc_auc_score(y_test, y_prob):.3f})")
plt.plot([0, 1], [0, 1], "--", color="#94A3B8", label="random guessing")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("ROC curve (held-out test set)")
plt.legend(loc="lower right")
plt.show()

### What drives a prediction?

Each coefficient as an **odds ratio**: how much a "yes" answer multiplies the odds of a
positive diagnosis, holding the other answers fixed.

In [ ]:
importance = m.feature_importances(pipeline)
importance[["question", "coefficient", "odds_ratio"]].round(3)

In [ ]:
ax = sns.barplot(data=importance, x="odds_ratio", y="question", color="#2563EB")
ax.axvline(1.0, color="#DC2626", linestyle="--", label="no effect")
ax.set_title("Odds ratio per answer ('yes' vs 'no')")
ax.set_xlabel("odds ratio")
ax.set_ylabel("")
ax.legend()
plt.show()

Every question pushes the same way — answering "yes" raises the odds of a diagnosis — and
`Chronic_Disease`, `Fatigue` and `Allergy` dominate. That is a property of the survey: its
questions are correlated symptom checkboxes rather than independent risk factors, so the
coefficients should not be read as causal risk.

## 6. Save the model

This writes the three files the app loads, so the app is always in sync with this notebook.

In [ ]:
metadata = m.build_metadata(clean, metrics)
m.save_artifacts(pipeline, metadata, clean)
print("wrote:", m.CLEAN_DATA.name, "|", m.MODEL_FILE.name, "|", m.METADATA_FILE.name)

In [ ]:
profile = dict(Age=45, Gender="Male", **{col: "No" for col in m.BINARY_FEATURES})
print("all-clear profile      :", m.predict_one(pipeline, profile))

profile.update(Chronic_Disease="Yes", Fatigue="Yes", Allergy="Yes")
print("three symptom answers  :", m.predict_one(pipeline, profile))

profile.update(Smoking="Yes", Coughing="Yes", Wheezing="Yes")
print("six symptom answers    :", m.predict_one(pipeline, profile))

## 7. Conclusion and limits

The model predicts the survey's yes/no diagnosis with **ROC AUC 0.97** on held-out data and
balanced accuracy 0.93, comfortably above the 0.855 "always yes" baseline.

Worth remembering:

* **The dataset is tiny** — 276 rows after de-duplication, so only 10 negative cases land in
  the test set. The confidence intervals around those numbers are wide.
* **The split is small and fixed** (`random_state=42`). Re-running with a different seed moves
  the metrics by several points; the 5-fold CV figures are the more stable ones.
* **Class weighting shifts probabilities.** They are useful for ranking, but they are not a
  calibrated clinical risk. Treat them as a relative score.
* **Survey data is self-reported and this is not a medical device.** It is a modelling
  exercise, not a diagnosis.

**Run the app:**

```bash
streamlit run app.py
```